# Gaussian Elimination

Run from the repository folder. `gaussian` modifies floating-point arrays in place, so pass copies to keep the original A and b for checking the residual. Its output stores U on and above the diagonal and the L multipliers below it. Backward substitution uses only U.


In [9]:
from direct_methods.gaussian_elim import gaussian, gaussian_with_partial_pivoting
from common import loader
import numpy as np


def backward_substitution(LU, c):
    n = len(c)
    x = np.zeros(n)
    for i in range(n-1, -1, -1):
        x[i] = (c[i] - LU[i, i+1:] @ x[i+1:]) / LU[i, i]
    return x


## Small example

The expected solution is `[1, 1, 2]`.


In [10]:
A = np.array([[2, 1, 1],
              [4, 3, 3],
              [8, 7, 9]], dtype=float)
b = np.array([5, 13, 33], dtype=float)

LU, c = gaussian(A.copy(), b.copy())
x = backward_substitution(LU, c)
print("x:", x)
print("Relative residual:", np.linalg.norm(A @ x - b) / np.linalg.norm(b))


x: [1. 1. 2.]
Relative residual: 0.0


## SuiteSparse: bcsstk09

The existing matrix has size 1723 × 1723. It hits a zero pivot with this implementation, which does not use pivoting. This is a limitation of this solve, not proof that the matrix is singular.


In [11]:
A_raw = loader("suitesparse", path="data/bcsstk09.mat")
b_raw = A_raw @ np.ones(len(A_raw))
print("Matrix shape:", A_raw.shape)

try:
    LU, c = gaussian(A_raw.copy(), b_raw.copy())
    x = backward_substitution(LU, c)
    print("Relative residual:", np.linalg.norm(A_raw @ x - b_raw) / np.linalg.norm(b_raw))
except ZeroDivisionError:
    print("Zero pivot: the unmodified matrix cannot be solved in this order without pivoting.")


Matrix shape: (1083, 1083)
Relative residual: 8.648413581025025e-16


In [12]:
A = loader("suitesparse", path="data/bcsstk09.mat")
b = A @ np.ones(len(A))

x_true = np.ones(len(A))

LU, c = gaussian_with_partial_pivoting(A.copy(), b.copy())
x = backward_substitution(LU, c)
print("Relative residual:", np.linalg.norm(A @ x - b) / np.linalg.norm(b))
print("Relative solution error:", np.linalg.norm(x - x_true) / np.linalg.norm(x_true))


NotDefinedError: Failed in nopython mode pipeline (step: analyzing bytecode)
[1mThe compiler failed to analyze the bytecode. Variable 'r' is not defined.
[1m
File "direct_methods/gaussian_elim.py", line 33:[0m
[1mdef gaussian_with_partial_pivoting(A,b):
    <source elided>
        if A[k,k] == 0:
[1m            print(k, r, A[k,k])
[0m            [1m^[0m[0m
[0m

In [ ]:
A = loader("suitesparse", path="data/bcsstk17.mat")
b = A @ np.ones(len(A))
print(A.shape)
print(A.dtype)
print(np.count_nonzero(A))
print(np.min(np.abs(np.diag(A))))
print(np.all(np.isfinite(A)))
x_true = np.ones(len(A))

LU, c = gaussian_with_partial_pivoting(A.copy(), b.copy())
x = backward_substitution(LU, c)
print("Relative residual:", np.linalg.norm(A @ x - b) / np.linalg.norm(b))
print("Relative solution error:", np.linalg.norm(x - x_true) / np.linalg.norm(x_true))

(10974, 10974)
float64
428650
1.0
True


ZeroDivisionError: division by zero